# PART 03 · 3장 분류 모델링 — 연습 문제 정답

교재 259쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

multi_base = 'data/part03_type2/multiclass_ticket_priority'
multi_train = pd.read_csv(multi_base + '/train.csv')

y_multi = multi_train['priority_level'].copy()
X_multi_raw = multi_train.drop(
    columns=['priority_level', 'ticket_id']
).copy()

X_multi_train_raw, X_multi_val_raw, y_multi_train, y_multi_val = train_test_split(
    X_multi_raw, y_multi, test_size=0.2, random_state=0, stratify=y_multi
)

num_cols = X_multi_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_multi_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_multi_train_raw[col].median()
    X_multi_train_raw[col] = X_multi_train_raw[col].fillna(fill_value)
    X_multi_val_raw[col] = X_multi_val_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_multi_train_raw[col].mode()[0]
    X_multi_train_raw[col] = X_multi_train_raw[col].fillna(fill_value)
    X_multi_val_raw[col] = X_multi_val_raw[col].fillna(fill_value)

X_multi_train = pd.get_dummies(X_multi_train_raw)
X_multi_val = pd.get_dummies(X_multi_val_raw).reindex(
    columns=X_multi_train.columns, fill_value=0
)

multi_model = RandomForestClassifier(n_estimators=100, random_state=0)
multi_model.fit(X_multi_train, y_multi_train)
multi_pred = multi_model.predict(X_multi_val)
print(X_multi_train.shape, X_multi_val.shape)

(624, 28) (156, 28)


## 문제 01

predict_proba( ) 결과의 각 열이 어떤 클래스에 해당하는지 확인하는 속성은 무엇인가?

**정답 코드**

In [2]:
proba = multi_model.predict_proba(X_multi_val)

print('classes_:', multi_model.classes_)
print('proba shape:', proba.shape)
print('첫 행의 확률:', proba[0].round(4))
print('첫 행 확률 합:', proba[0].sum())

classes_: [0 1 2]
proba shape: (156, 3)
첫 행의 확률: [0.37 0.56 0.07]
첫 행 확률 합: 1.0


**간단 해설**

predict_proba()의 열 순서는 model.classes_에 저장된 클래스 순서와 같다.

## 문제 02

Macro F1이 Weighted F1보다 낮게 나올 수 있는 이유로 가장 적절한 것은 무엇인가?

**정답 코드**

In [3]:
f1_each = f1_score(y_multi_val, multi_pred, average=None)
macro = f1_score(y_multi_val, multi_pred, average='macro')
weighted = f1_score(y_multi_val, multi_pred, average='weighted')

print('클래스별 F1:', f1_each.round(4))
print('클래스별 개수:', y_multi_val.value_counts().sort_index().tolist())
print('Macro F1:', round(macro, 4))
print('Weighted F1:', round(weighted, 4))

클래스별 F1: [0.5968 0.6335 0.1481]
클래스별 개수: [61, 72, 23]
Macro F1: 0.4595
Weighted F1: 0.5476


**간단 해설**

Macro F1은 클래스별 데이터 수를 고려하지 않는다. 데이터가 적은 클래스라도 성능이 낮으면 전체 점수가 크게 낮아질 수 있다.

---

**정답** 1. model.classes_, 2. 소수 클래스의 낮은 F1-score도 같은 비중으로 반영되기 때문이다.